In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.data import Data
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset, DataLoader
from google.colab import userdata
import kagglehub

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kagglehub torch torch-geometric transformers sentence-transformers scikit-learn faiss-cpu pandas numpy tqdm

path = kagglehub.competition_download('recommender-system-challenge-2526-s-2')

sample_submission_df = pd.read_csv(f'{path}/sample_submission.csv')
train_df = pd.read_csv(f'{path}/train.csv')
test_df = pd.read_csv(f'{path}/test.csv')
metadata_df = pd.read_csv(f'{path}/item_meta.csv')

train_df['timestamp'] = pd.to_datetime(train_df['timestamp'], unit='ms')
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'], unit='ms')

target_users = sample_submission_df['user_id'].unique()

train_df = train_df.sort_values('timestamp')
cutoff_time = train_df['timestamp'].quantile(0.8)
train_data = train_df[train_df['timestamp'] <= cutoff_time]
val_data = train_df[train_df['timestamp'] > cutoff_time]

all_item_ids = set(train_df['item_id'].unique()) | set(metadata_df['item_id'].unique())
all_user_ids = train_df['user_id'].unique()

item_id_to_idx = {item_id: idx for idx, item_id in enumerate(sorted(all_item_ids))}
idx_to_item_id = {idx: item_id for item_id, idx in item_id_to_idx.items()}
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(sorted(all_user_ids))}
idx_to_user_id = {idx: user_id for user_id, idx in user_id_to_idx.items()}

num_items = len(all_item_ids)
num_users = len(all_user_ids)

item_counts = train_df['item_id'].value_counts()
popular_items = item_counts.head(200).index.tolist()

MAX_SEQ_LEN = 50

def create_user_sequences(df, max_seq_len):
    df = df.sort_values(['user_id', 'timestamp'])
    user_sequences = df.groupby('user_id')['item_id'].apply(list).reset_index()
    sequences = []
    for _, row in user_sequences.iterrows():
        seq = row['item_id']
        seq = seq[-max_seq_len:] if len(seq) > max_seq_len else seq
        for i in range(1, len(seq)):
            sequences.append((row['user_id'], seq[:i], seq[i]))
    return sequences

train_sequences = create_user_sequences(train_data, MAX_SEQ_LEN)
val_sequences = create_user_sequences(val_data, MAX_SEQ_LEN)

sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

all_items_df = pd.DataFrame({'item_id': list(all_item_ids)})
metadata_df = metadata_df.set_index('item_id')
all_items_df = all_items_df.join(metadata_df, on='item_id', how='left').reset_index()

all_items_df.fillna({
    'title': '', 'features': '', 'description': '', 'main_category': 'Unknown',
    'store': 'Unknown', 'price': 0, 'average_rating': 0, 'rating_number': 0
}, inplace=True)

def create_item_text(row):
    return f"{row['title']} {row['features']} {row['description']} {row['main_category']} {row['store']}"

all_items_df['text'] = all_items_df.apply(create_item_text, axis=1)

BATCH_SIZE_SBERT = 256
item_embeddings = []
for i in tqdm(range(0, len(all_items_df), BATCH_SIZE_SBERT), desc="SBERT Embeddings"):
    batch = all_items_df['text'].iloc[i:i+BATCH_SIZE_SBERT].tolist()
    embeddings = sbert_model.encode(batch, convert_to_tensor=False, show_progress_bar=False)
    item_embeddings.append(embeddings)
item_embeddings = np.concatenate(item_embeddings, axis=0)

NUMERIC_COLS = ['price', 'average_rating', 'rating_number']
scaler = MinMaxScaler()
all_items_df[NUMERIC_COLS] = scaler.fit_transform(all_items_df[NUMERIC_COLS])

CAT_COLS = ['main_category', 'store']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
cat_features = encoder.fit_transform(all_items_df[CAT_COLS])

content_features = np.hstack([item_embeddings, all_items_df[NUMERIC_COLS].values, cat_features.toarray()])

class CustomLightGCN(MessagePassing):
    def __init__(self, alpha=1.0):
        super().__init__(aggr='add')
        self.alpha = alpha

    def forward(self, x, edge_index):
        row, col = edge_index
        deg = degree(row, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        return self.propagate(edge_index, x=x, norm=norm)

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j * self.alpha

    def update(self, aggr_out):
        return aggr_out

class LightGCNModel(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, num_layers=3, alpha=1.0):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        self.lightgcn_layers = nn.ModuleList([CustomLightGCN(alpha=alpha) for _ in range(num_layers)])

    def forward(self, data):
        x = self.embedding(torch.arange(data.num_nodes, device=self.embedding.weight.device))
        for layer in self.lightgcn_layers:
            x = layer(x, data.edge_index)
        return x

    def get_embeddings(self, data):
        with torch.no_grad():
            return self.forward(data)

def build_graph(df, num_users, num_items, user_id_to_idx, item_id_to_idx):
    valid_items = set(item_id_to_idx.keys())
    valid_users = set(user_id_to_idx.keys())
    filtered_df = df[df['item_id'].isin(valid_items) & df['user_id'].isin(valid_users)]
    
    user_item_pairs = filtered_df[['user_id', 'item_id']].drop_duplicates().values
    user_indices = [user_id_to_idx[uid] for uid in user_item_pairs[:, 0]]
    item_indices = [item_id_to_idx[iid] for iid in user_item_pairs[:, 1]]

    user_item_edges = torch.tensor([user_indices, item_indices], dtype=torch.long)
    item_user_edges = torch.stack([user_item_edges[1], user_item_edges[0]])
    all_edges = torch.cat([user_item_edges, item_user_edges], dim=1)

    return Data(edge_index=all_edges, num_nodes=num_users + num_items)

graph = build_graph(train_data, num_users, num_items, user_id_to_idx, item_id_to_idx)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
graph = graph.to(device)

lightgcn_model = LightGCNModel(num_users + num_items, embedding_dim=64, num_layers=3).to(device)
optimizer_lightgcn = torch.optim.Adam(lightgcn_model.parameters(), lr=0.001)
scheduler_lightgcn = ReduceLROnPlateau(optimizer_lightgcn, mode='min', patience=3)

best_lightgcn_loss = float('inf')
patience_counter = 0
MAX_PATIENCE = 5

def train_lightgcn(model, graph, epochs=50):
    global best_lightgcn_loss, patience_counter
    model.train()
    for epoch in range(epochs):
        optimizer_lightgcn.zero_grad()
        embeddings = model(graph)

        pos_edges = graph.edge_index[:, :graph.edge_index.shape[1]//2]
        neg_edges = torch.randint(0, num_items, (2, pos_edges.shape[1]), device=device)
        neg_edges[0] = pos_edges[0]

        pos_emb = embeddings[pos_edges[0]] * embeddings[pos_edges[1]]
        neg_emb = embeddings[neg_edges[0]] * embeddings[neg_edges[1]]
        pos_scores = torch.sum(pos_emb, dim=1)
        neg_scores = torch.sum(neg_emb, dim=1)

        loss = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
        loss.backward()
        optimizer_lightgcn.step()
        scheduler_lightgcn.step(loss)

        if loss.item() < best_lightgcn_loss:
            best_lightgcn_loss = loss.item()
            patience_counter = 0
            torch.save(model.state_dict(), 'best_lightgcn.pt')
        else:
            patience_counter += 1
            if patience_counter >= MAX_PATIENCE:
                model.load_state_dict(torch.load('best_lightgcn.pt'))
                break

train_lightgcn(lightgcn_model, graph, epochs=50)

lightgcn_model.eval()
lightgcn_embeddings = lightgcn_model.get_embeddings(graph).cpu().numpy()
user_embeddings_lightgcn = lightgcn_embeddings[:num_users]
item_embeddings_lightgcn = lightgcn_embeddings[num_users:]

class BERT4Rec(nn.Module):
    def __init__(self, num_items, embedding_dim=128, num_heads=4, num_layers=2, max_seq_len=50):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embedding_dim)
        self.dropout = nn.Dropout(0.2)
        self.max_seq_len = max_seq_len
        self.num_items = num_items

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads, dim_feedforward=embedding_dim * 4,
            dropout=0.2, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embedding_dim, num_items)

    def forward(self, input_seqs):
        batch_size, seq_len = input_seqs.shape
        
        if seq_len > self.max_seq_len or (input_seqs >= self.num_items).any():
            raise ValueError("Invalid sequence length or indices")

        seq_emb = self.item_embedding(input_seqs)
        pos_ids = torch.arange(seq_len, dtype=torch.long, device=input_seqs.device).unsqueeze(0).expand(batch_size, -1)
        pos_emb = self.position_embedding(pos_ids)
        x = self.dropout(seq_emb + pos_emb)
        
        x = self.transformer(x)
        return self.fc(x[:, -1, :])

class SeqDataset(Dataset):
    def __init__(self, sequences, item_id_to_idx, max_seq_len):
        self.sequences = sequences
        self.item_id_to_idx = item_id_to_idx
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        _, seq, target = self.sequences[idx]
        seq_indices = [self.item_id_to_idx.get(item_id, 0) for item_id in seq[-self.max_seq_len:]]
        target_idx = self.item_id_to_idx.get(target, 0)

        seq_indices = (seq_indices + [0] * self.max_seq_len)[:self.max_seq_len]
        return torch.tensor(seq_indices, dtype=torch.long), torch.tensor(target_idx, dtype=torch.long)

dataset = SeqDataset(train_sequences, item_id_to_idx, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, pin_memory=True)

bert4rec_model = BERT4Rec(num_items, embedding_dim=128, num_heads=4, num_layers=2, max_seq_len=MAX_SEQ_LEN).to(device)
optimizer_bert4rec = torch.optim.AdamW(bert4rec_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler_bert4rec = ReduceLROnPlateau(optimizer_bert4rec, mode='min', patience=3)
criterion = nn.CrossEntropyLoss()

best_bert4rec_loss = float('inf')
patience_counter_bert = 0

def train_bert4rec(model, dataloader, epochs=20):
    global best_bert4rec_loss, patience_counter_bert
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_seqs, batch_targets in tqdm(dataloader, desc=f"BERT4Rec Epoch {epoch+1}"):
            batch_seqs, batch_targets = batch_seqs.to(device), batch_targets.to(device)
            optimizer_bert4rec.zero_grad()
            logits = model(batch_seqs)
            loss = criterion(logits, batch_targets)
            loss.backward()
            optimizer_bert4rec.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        scheduler_bert4rec.step(avg_loss)

        if avg_loss < best_bert4rec_loss:
            best_bert4rec_loss = avg_loss
            patience_counter_bert = 0
            torch.save(model.state_dict(), 'best_bert4rec.pt')
        else:
            patience_counter_bert += 1
            if patience_counter_bert >= 5:
                model.load_state_dict(torch.load('best_bert4rec.pt'))
                break

train_bert4rec(bert4rec_model, dataloader, epochs=20)

class ColdStartModel:
    def __init__(self, content_features, item_ids, popular_items):
        self.content_features = content_features
        self.item_ids = item_ids
        self.popular_items = popular_items

    def recommend_for_user(self, user_history, top_k=10):
        if len(user_history) > 0:
            history_indices = [idx for item_id in user_history if (idx := item_id_to_idx.get(item_id, -1)) != -1]
            if len(history_indices) > 0:
                user_embedding = np.mean(self.content_features[history_indices], axis=0)
                scores = cosine_similarity([user_embedding], self.content_features)[0]
            else:
                scores = np.zeros(len(self.item_ids))
        else:
            scores = np.zeros(len(self.item_ids))
            for item_id in self.popular_items:
                if item_id in self.item_ids:
                    scores[list(self.item_ids).index(item_id)] = 1.0

        top_indices = np.argsort(-scores)[:top_k * 2]
        recommendations = []
        for idx in top_indices:
            item_id = self.item_ids[idx]
            if item_id not in user_history:
                recommendations.append(item_id)
                if len(recommendations) == top_k:
                    break
        return recommendations

cold_start_model = ColdStartModel(content_features, list(item_id_to_idx.keys()), popular_items)

def retrieve_candidates(user_id, user_history, top_k=100):
    user_idx = user_id_to_idx.get(user_id, -1)
    if user_idx == -1:
        candidates = cold_start_model.recommend_for_user(user_history, top_k=top_k)
        return [item_id_to_idx.get(item_id, -1) for item_id in candidates]

    user_emb = user_embeddings_lightgcn[user_idx]
    lightgcn_scores = np.dot(user_emb, item_embeddings_lightgcn.T)

    if len(user_history) > 0:
        history_indices = [idx for item_id in user_history if (idx := item_id_to_idx.get(item_id, -1)) != -1]
        if len(history_indices) > 0:
            history_emb = np.mean(content_features[history_indices], axis=0)
            content_scores = cosine_similarity([history_emb], content_features)[0]
        else:
            content_scores = np.zeros(num_items)
    else:
        content_scores = np.zeros(num_items)

    combined_scores = best_alpha * lightgcn_scores + best_beta * content_scores
    candidate_indices = np.argsort(-combined_scores)[:top_k]
    return [int(idx) for idx in candidate_indices if 0 <= idx < num_items]

def rerank_candidates(user_id, candidate_indices, user_history, top_k=10):
    if len(candidate_indices) == 0:
        return cold_start_model.recommend_for_user(user_history, top_k=top_k)

    user_history_indices = [item_id_to_idx[item_id] for item_id in user_history[-MAX_SEQ_LEN:] if item_id in item_id_to_idx]
    user_history_indices = (user_history_indices[:MAX_SEQ_LEN - 1] + [0] * (MAX_SEQ_LEN - 1))[:MAX_SEQ_LEN - 1]

    input_seq = torch.tensor(user_history_indices, dtype=torch.long).unsqueeze(0).to(device)

    if (input_seq >= num_items).any():
        return cold_start_model.recommend_for_user(user_history, top_k=top_k)

    bert4rec_model.eval()
    with torch.no_grad():
        try:
            logits = bert4rec_model(input_seq)
        except Exception:
            return cold_start_model.recommend_for_user(user_history, top_k=top_k)

    logits = logits.squeeze().cpu().numpy()
    candidate_indices = [idx for idx in candidate_indices if 0 <= idx < num_items]
    
    if len(candidate_indices) == 0:
        return cold_start_model.recommend_for_user(user_history, top_k=top_k)

    candidate_scores = logits[candidate_indices]
    reranked_indices = np.argsort(-candidate_scores)[:top_k]
    return [idx_to_item_id[candidate_indices[i]] for i in reranked_indices]

best_recall = 0
best_alpha = 0.7
best_beta = 0.3

for alpha in [0.6, 0.65, 0.7, 0.75, 0.8]:
    beta = 1.0 - alpha
    hits = 0
    total = 0
    for user_id, input_seq, target in val_sequences[:500]:
        user_idx = user_id_to_idx.get(user_id, -1)
        if user_idx == -1:
            continue

        user_emb = user_embeddings_lightgcn[user_idx]
        lightgcn_scores = np.dot(user_emb, item_embeddings_lightgcn.T)

        if len(input_seq) > 0:
            history_indices = [idx for item_id in input_seq if (idx := item_id_to_idx.get(item_id, -1)) != -1]
            if len(history_indices) > 0:
                history_emb = np.mean(content_features[history_indices], axis=0)
                content_scores = cosine_similarity([history_emb], content_features)[0]
            else:
                content_scores = np.zeros(num_items)
        else:
            content_scores = np.zeros(num_items)

        combined_scores = alpha * lightgcn_scores + beta * content_scores
        candidate_indices = np.argsort(-combined_scores)[:100]

        if item_id_to_idx.get(target, -1) in candidate_indices[:10]:
            hits += 1
        total += 1

    current_recall = hits / total if total > 0 else 0
    if current_recall > best_recall:
        best_recall = current_recall
        best_alpha, best_beta = alpha, beta

print(f"Optimized weights: alpha={best_alpha}, beta={best_beta} (Recall@10={best_recall:.6f})")

def evaluate_recall_at_10(model, val_sequences, top_k=10):
    hits = 0
    total = 0
    for user_id, input_seq, target in val_sequences:
        input_seq_indices = [item_id_to_idx.get(item_id, 0) for item_id in input_seq[-MAX_SEQ_LEN:]]
        input_seq_tensor = torch.tensor(input_seq_indices, dtype=torch.long).unsqueeze(0).to(device)
        
        with torch.no_grad():
            logits = model(input_seq_tensor)
        pred_indices = torch.topk(logits, top_k).indices.squeeze().cpu().numpy()

        if item_id_to_idx.get(target, 0) in pred_indices:
            hits += 1
        total += 1
    return hits / total if total > 0 else 0.0

val_recall = evaluate_recall_at_10(bert4rec_model, val_sequences, top_k=10)
print(f"Recall@10 on temporal validation: {val_recall:.6f}")

def generate_submission(target_users, sample_submission_df):
    predictions = []
    for user_id in tqdm(target_users, desc="Generating predictions"):
        user_history = train_df[train_df['user_id'] == user_id]['item_id'].tolist()
        candidate_indices = retrieve_candidates(user_id, user_history, top_k=100)
        top_10 = rerank_candidates(user_id, candidate_indices, user_history, top_k=10)

        if len(top_10) < 10:
            top_10 += [item for item in popular_items if item not in top_10][:10 - len(top_10)]

        top_10 = [str(int(item)) for item in top_10 if int(item) in item_id_to_idx][:10]

        if len(top_10) < 10:
            top_10 += [str(item) for item in popular_items if str(item) not in top_10][:10 - len(top_10)]

        predictions.append(",".join(top_10))

    submission_df = sample_submission_df.copy()
    submission_df['item_id'] = predictions
    return submission_df

submission_df = generate_submission(target_users, sample_submission_df)
submission_df.to_csv('submission.csv', index=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 37.1 MB/s eta 0:00:00
